In [1]:
# =========================================================
# Notebook : Création du Silver Layer pour Telco Churn
# =========================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col, trim, lit
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

# ------------------------
# Configuration MinIO
# ------------------------
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "telco-churn"
MINIO_ENDPOINT = "minio1:9000"

# ------------------------
# Initialisation Spark avec ressources ajustées
# ------------------------
spark = (SparkSession.builder
         .appName("TelcoChurn_Silver")
         .master("spark://spark-master:7077")
         .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
         .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
         .config("spark.hadoop.fs.s3a.endpoint", f"http://{MINIO_ENDPOINT}")
         .config("spark.hadoop.fs.s3a.path.style.access", "true")
         .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
         .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
         # Ressources driver et executors
         .config("spark.driver.memory", "3g")
         .config("spark.executor.memory", "2g")
         .config("spark.executor.cores", "2")
         .config("spark.executor.instances", "2")
         # Shuffle partitions optimisé pour petit cluster
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ------------------------
# Lecture Bronze
# ------------------------
bronze_path = f"s3a://{MINIO_BUCKET}/bronze/telco_churn"
df = spark.read.format("delta").load(bronze_path)
print(f"✅ Bronze Layer chargé : {df.count()} lignes")

# ------------------------
# 1) Nettoyage TotalCharges
# ------------------------
df = (df
    .withColumn("TotalCharges", when(trim(col("TotalCharges")) == "", None)
                                 .otherwise(col("TotalCharges")))
    .withColumn("TotalCharges", col("TotalCharges").cast("double"))
    .na.fill({"TotalCharges": 0.0})
)

# ------------------------
# 2) Variables binaires
# ------------------------
df = df.withColumn("HasInternetService", when(col("InternetService") == "No", 0).otherwise(1))
df = df.withColumn("HasPhoneService", when(col("PhoneService") == "Yes", 1).otherwise(0))

service_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]
for c in service_cols:
    df = df.withColumn(c, when(col(c) == "Yes", 1).otherwise(0))

df = df.withColumn("MultipleLines", when(col("MultipleLines") == "Yes", 1).otherwise(0))

binary_cols = ["Partner", "Dependents", "PaperlessBilling", "Churn"]
for c in binary_cols:
    df = df.withColumn(c, when(col(c) == "Yes", 1).otherwise(0))

# ------------------------
# 3) Cast colonnes numériques
# ------------------------
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
for c in num_cols:
    df = df.withColumn(c, col(c).cast("double"))

# ------------------------
# 4) Encodage catégoriel (StringIndexer)
#    📌 Silver → seulement index, PAS OHE ici
# ------------------------
categorical_cols = ["Contract", "PaymentMethod"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
            for c in categorical_cols]

pipeline = Pipeline(stages=indexers)
df = pipeline.fit(df).transform(df)

# ------------------------
# 5) Sauvegarde Silver
# ------------------------
silver_path = f"s3a://{MINIO_BUCKET}/silver/telco_churn"
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_path)
print("✅ Silver Layer enregistré correctement :", silver_path)

df.show(5, truncate=False)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/17 18:59:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/17 18:59:41 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/12/17 18:59:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

✅ Bronze Layer chargé : 7041 lignes


✅ Silver Layer enregistré correctement : s3a://telco-churn/silver/telco_churn
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+-------------------------+--------------+------------+-----+------------------+---------------+------------+-----------------+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|Contract      |PaperlessBilling|PaymentMethod            |MonthlyCharges|TotalCharges|Churn|HasInternetService|HasPhoneService|Contract_idx|PaymentMethod_idx|
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+-----------

25/12/17 19:00:26 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
25/12/17 19:00:26 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:291)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:981)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:165)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:263)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:170)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:115)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.proce